### CASE TECNICO PYSPARK

In [2]:
##Import list
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

#set up global variables

CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

#start pyspark session

spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Set log level AFTER session creation
spark.sparkContext.setLogLevel("ERROR")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 21:57:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/25 21:57:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField

# explicit schemas avoid extra pass for inference
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Fast JSONL loader: schema-first, lazy, and optional repartition."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    return df


def load_data(dataframe: str) -> DataFrame:
    match dataframe:
        case "clientes":
            return load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
        case "pedidos":
            # larger file: increase parallelism for downstream transformations
            return load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)
        case _:
            raise ValueError(f"Unknown dataframe: {dataframe}")


# usage (lazy; no full scan here)
clientes_df = load_data("clientes")
pedidos_df= load_data("pedidos")

print("clientes partitions:", clientes_df.rdd.getNumPartitions())
print("pedidos partitions:", pedidos_df.rdd.getNumPartitions())
print("schemas loaded successfully")

clientes partitions: 1


pedidos partitions: 32
schemas loaded successfully


## 1. Data Quality - Relatório de Falhas

In [4]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Base columns (keep only what is needed)
pedidos_base_df = pedidos_df.select("id", "client_id", "value")
clientes_base_df = clientes_df.select("id", "name")

def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 1) Pedido sem valor
Sem_valor_ou_zero = regra_falha(
    pedidos_base_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    1
)

# 3) ID de pedido duplicado
ids_duplicados_df = (
    pedidos_base_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .select("id")
)
ids_duplicados = regra_falha(ids_duplicados_df, "id_duplicado", 3)

# 4) Pedido com cliente inexistente (somente client_id válido)
cliente_inexistente = (
    pedidos_base_df.alias("p")
    .filter(F.col("p.client_id").isNotNull() & (F.col("p.client_id") > 0))
    .join(
        broadcast(clientes_base_df.select(F.col("id").alias("client_id_ref")).alias("c")),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left_anti"
    )
    .select(F.col("p.id").alias("id"))
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 4))
)

# 5) ID nulo
id_nullo = regra_falha(
    pedidos_base_df.filter(F.col("id").isNull()),
    "id_nulo",
    5
)

# 6) client_id nulo
client_id_nulo = regra_falha(
    pedidos_base_df.filter(F.col("client_id").isNull()),
    "client_id_nulo",
    6
)

# 7) ID inválido (<= 0)
id_invalido = regra_falha(
    pedidos_base_df.filter(F.col("id").isNotNull() & (F.col("id") <= 0)),
    "id_invalido_menor_igual_zero",
    7
)

# 8) client_id inválido (<= 0)
client_id_invalido = regra_falha(
    pedidos_base_df.filter(F.col("client_id").isNotNull() & (F.col("client_id") <= 0)),
    "client_id_invalido_menor_igual_zero",
    8
)

# 9) Valor zero
valor_zero = regra_falha(
    pedidos_base_df.filter(F.col("value") == 0),
    "valor_zero",
    9
)

# 10) Retorno sem pedido original correspondente
# Regra: para value < 0, deve existir (mesmo client_id, value positivo igual ao valor absoluto)
retornos_df = (
    pedidos_base_df
    .filter(F.col("value") < 0)
    .select(
        "id",
        "client_id",
        F.abs(F.col("value")).alias("valor_absoluto")
    )
)

pedidos_positivos_ref_df = (
    pedidos_base_df
    .filter(F.col("value") > 0)
    .select(
        F.col("client_id").alias("client_id_ref"),
        F.col("value").alias("valor_ref")
    )
    .distinct()
)

retornos_invalidos = (
    retornos_df.alias("r")
    .join(
        pedidos_positivos_ref_df.alias("p"),
        (F.col("r.client_id") == F.col("p.client_id_ref")) &
        (F.col("r.valor_absoluto") == F.col("p.valor_ref")),
        "left_anti"
    )
    .select(F.col("r.id").alias("id"))
    .transform(lambda df: regra_falha(df, "retorno_sem_pedido_original", 10))
)

# União de todas as regras
regras_falhas = [
    Sem_valor_ou_zero, ids_duplicados, cliente_inexistente, id_nullo, client_id_nulo,
    id_invalido, client_id_invalido, valor_zero, retornos_invalidos
]

falhas_df = (
    reduce(lambda acc, d: acc.unionByName(d), regras_falhas)
    .dropDuplicates(["id", "motivo"])
    .orderBy("ordem_regra", "id")
    .select("id", "motivo")
)

# Saídas pedidas
falhas_df.show(100, truncate=False)

erros_por_categoria_df = (
    falhas_df
    .groupBy("motivo")
    .agg(F.count("*").alias("qtd_erros"))
    .orderBy(F.col("qtd_erros").desc(), F.col("motivo").asc())
)

erros_por_categoria_df.show(truncate=False)
falhas_df.agg(F.count("*").alias("total_erros")).show()


+------+----------------+
|id    |motivo          |
+------+----------------+
|1534  |pedido_sem_valor|
|3502  |pedido_sem_valor|
|3679  |pedido_sem_valor|
|4322  |pedido_sem_valor|
|4724  |pedido_sem_valor|
|5774  |pedido_sem_valor|
|6322  |pedido_sem_valor|
|6622  |pedido_sem_valor|
|6884  |pedido_sem_valor|
|7116  |pedido_sem_valor|
|8597  |pedido_sem_valor|
|9322  |pedido_sem_valor|
|10786 |pedido_sem_valor|
|14240 |pedido_sem_valor|
|14804 |pedido_sem_valor|
|16194 |pedido_sem_valor|
|16383 |pedido_sem_valor|
|16929 |pedido_sem_valor|
|18559 |pedido_sem_valor|
|18623 |pedido_sem_valor|
|19401 |pedido_sem_valor|
|22785 |pedido_sem_valor|
|23468 |pedido_sem_valor|
|25638 |pedido_sem_valor|
|28249 |pedido_sem_valor|
|30342 |pedido_sem_valor|
|30377 |pedido_sem_valor|
|31027 |pedido_sem_valor|
|31045 |pedido_sem_valor|
|39954 |pedido_sem_valor|
|41080 |pedido_sem_valor|
|44260 |pedido_sem_valor|
|44409 |pedido_sem_valor|
|48490 |pedido_sem_valor|
|48920 |pedido_sem_valor|
|53800 |pedi

+-----------------------------------+---------+
|motivo                             |qtd_erros|
+-----------------------------------+---------+
|id_duplicado                       |54491    |
|pedido_sem_valor                   |49985    |
|retorno_sem_pedido_original        |24741    |
|client_id_invalido_menor_igual_zero|48       |
+-----------------------------------+---------+



+-----------+
|total_erros|
+-----------+
|     129265|
+-----------+



In [9]:
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Base: only the columns we need; keep partitions aligned with the client join and cache for reuse
pedidos_base_df = (
    pedidos_df
    .select("id", "client_id", "value")
    .repartition(64, "client_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
total_pedidos = pedidos_base_df.count()

clientes_ids_df = (
    clientes_df
    .select(F.col("id").alias("client_id_ref"))
    .distinct()
)

# Aux 1: duplicated ids (any id that appears more than once)
ids_duplicados_df = (
    pedidos_base_df
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Aux 2: positive orders to validate returns (value < 0)
pedidos_positivos_ref_df = (
    pedidos_base_df
    .filter(F.col("value") > 0)
    .dropDuplicates(["client_id", "value"])
    .select(
        F.col("client_id").alias("ret_client_id_ref"),
        F.col("value").alias("ret_valor_ref"),
    )
)

# Enrich once with all info needed for filtering
enriched_pedidos_df = (
    pedidos_base_df.alias("p")
    # mark duplicated ids (broadcast small helper)
    .join(
        broadcast(ids_duplicados_df).alias("d"),
        F.col("p.id") == F.col("d.dup_id"),
        "left",
    )
    # attach client existence
    .join(
        broadcast(clientes_ids_df).alias("c"),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left",
    )
    # attach matching positive order for potential returns
    .join(
        pedidos_positivos_ref_df.alias("rref"),
        (F.col("p.client_id") == F.col("rref.ret_client_id_ref"))
        & (F.abs(F.col("p.value")) == F.col("rref.ret_valor_ref")),
        "left",
    )
)

# Keep only rows that pass all quality rules
pedidos_validos_df = (
    enriched_pedidos_df
    .filter(
        # value present, non-zero, and within decimal(5,2) range
        F.col("p.value").isNotNull()
        & (F.col("p.value") != 0)
        & (F.col("p.value") >= F.lit(-999.99))
        & (F.col("p.value") <= F.lit(999.99))
        # valid id and client_id (not null, > 0)
        & F.col("p.id").isNotNull()
        & (F.col("p.id") > 0)
        & F.col("p.client_id").isNotNull()
        & (F.col("p.client_id") > 0)
        # not duplicated id
        & F.col("d.dup_id").isNull()
        # client exists for valid client_id
        & F.col("c.client_id_ref").isNotNull()
        # if value < 0, must have matching positive order; otherwise ok
        & (
            (F.col("p.value") >= 0)
            | (F.col("rref.ret_client_id_ref").isNotNull())
        )
    )
    .select(
        F.col("p.id").alias("id"),
        F.col("p.client_id").alias("client_id"),
        F.col("p.value").alias("value"),
    )
)
pedidos_validos_df = pedidos_validos_df.persist(StorageLevel.MEMORY_AND_DISK)
pedidos_validos = pedidos_validos_df.count()

print("Total pedidos (cached base):", total_pedidos)
print("Pedidos válidos (sem falhas nas regras):", pedidos_validos)

Total pedidos (cached base): 1100000
Pedidos válidos (sem falhas nas regras): 940449


In [ ]:
# Requisito 2: agregação por cliente a partir de pedidos válidos (considerando devoluções)
client_totals_with_returns_df = (
    pedidos_validos_df.alias("p")
    .groupBy("client_id")
    .agg(
        F.sum(F.when(F.col("p.value") > 0, F.lit(1)).otherwise(F.lit(0))).cast(LongType()).alias("qtd_pedidos"),
        F.sum(F.when(F.col("p.value") < 0, F.lit(1)).otherwise(F.lit(0))).cast(LongType()).alias("qtd_devolucoes"),
        F.sum(F.when(F.col("p.value") > 0, F.col("p.value")).otherwise(F.lit(0))).alias("valor_vendido"),
        F.sum(F.when(F.col("p.value") < 0, F.col("p.value")).otherwise(F.lit(0))).alias("valor_devolvido"),
    )
    .withColumn(
        "valor_total",
        (F.col("valor_vendido") + F.col("valor_devolvido")).cast(DecimalType(11, 2))
    )
    .drop("valor_vendido", "valor_devolvido")
)

clientes_totais_df = (
    client_totals_with_returns_df.alias("tot")
    .join(
        broadcast(clientes_df.select("id", "name").alias("c")),
        F.col("tot.client_id") == F.col("c.id"),
        "left",
    )
    .select(
        F.col("c.name").alias("nome_cliente"),
        F.col("tot.qtd_pedidos"),
        F.col("tot.qtd_devolucoes"),
        F.col("tot.valor_total"),
    )
    .orderBy(F.col("tot.valor_total").desc(), F.col("nome_cliente"))
)

clientes_totais_df = clientes_totais_df.persist(StorageLevel.MEMORY_AND_DISK)
clientes_totais_df.show(50, truncate=False)


+---------------+-----------+--------------+-----------+
|   nome_cliente|qtd_pedidos|qtd_devolucoes|valor_total|
+---------------+-----------+--------------+-----------+
|  Inês Siqueira|     469734|             0|23698016.90|
|   Zachary Reis|         61|             0|    4002.08|
|  Vitor Marques|         68|             0|    3937.06|
|    Wanda Silva|         61|             0|    3736.53|
|  Inês Siqueira|         60|             0|    3735.58|
|    Tereza Leal|         66|             0|    3731.10|
|   Mariana Melo|         71|             0|    3700.71|
|Yasmin Carvalho|         65|             0|    3679.35|
|    Tereza Leal|         67|             0|    3678.17|
| Gustavo Pontes|         65|             0|    3657.84|
|   Sofia Castro|         66|             0|    3656.86|
|Vitória Andrade|         69|             0|    3656.43|
|   Breno Soares|         66|             0|    3651.20|
|   João Batista|         68|             0|    3650.20|
|    Julio Viana|         62|  